In [1]:
"""Building a Rack System with LangChain and FAISS

Introduction to RAG (Recreable Augmented Generation) RAG combines the power of retrieval systems with generative AI models instead of relying solely on the model's training data.
1. RAG first retrieves relevant documents from the knowledge base.
2. Second, uses these documents as context for the LLM.
3. Third, generates responses based on both the retrieved context and the model's knowledge."""

"Building a Rack System with LangChain and FAISS\n\nIntroduction to RAG (Recreable Augmented Generation) RAG combines the power of retrieval systems with generative AI models instead of relying solely on the model's training data.\n1. RAG first retrieves relevant documents from the knowledge base.\n2. Second, uses these documents as context for the LLM.\n3. Third, generates responses based on both the retrieved context and the model's knowledge."

In [2]:
"""FAISS (Facebook AI Similarity Search) is a library for efficient similarity search and clustering of dense vectors."""

'FAISS (Facebook AI Similarity Search) is a library for efficient similarity search and clustering of dense vectors.'

In [3]:
"""Key Advantages of FAISS:
1. High accuracy
2. Fast retrieval
3. Scalable
4. Easy to use
5. Open-source"""


'Key Advantages of FAISS:\n1. High accuracy\n2. Fast retrieval\n3. Scalable\n4. Easy to use\n5. Open-source'

In [4]:
## How it works:
"""1. Indexes vectors for fast nearest neighbor search
2. Returns most similar vectors based on distance metrics"""

'1. Indexes vectors for fast nearest neighbor search\n2. Returns most similar vectors based on distance metrics'

In [7]:
# load libraries
import os
from dotenv import load_dotenv
import numpy as np
from typing import List, Dict, Any, Optional
import warnings
warnings.filterwarnings("ignore")


# LangChain core imports
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import (
    RunnablePassthrough,
)
from langchain_core.output_parsers import StrOutputParser


# Langchain Specific imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain


# load environment variable
load_dotenv()

True

In [8]:
## Data Ingestion and Processing

In [9]:
sample_documents = [
    Document(
        page_content="""
        Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into narrow AI and general AI.
        """,
        metadata={"source": "AI Introduction", "page": 1, "topic": "AI"}
    ),
    Document(
        page_content="""
        Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised, unsupervised, and reinforcement learning.
        """,
        metadata={"source": "ML Basics", "page": 1, "topic": "ML"}
    ),
    Document(
        page_content="""
        Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses multiple layers to progressively extract higher-level features from raw input.
        Deep learning has revolutionized computer vision, NLP, and speech recognition.
        """,
        metadata={"source": "Deep Learning", "page": 1, "topic": "DL"}
    ),
    Document(
        page_content="""
        Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.
        It combines computational linguistics with machine learning and deep learning models.
        Applications include chatbots, translation, sentiment analysis, and text summarization.
        """,
        metadata={"source": "NLP Overview", "page": 1, "topic": "NLP"}
    )
]

print(sample_documents)

[Document(metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}, page_content='\n        Artificial Intelligence (AI) is the simulation of human intelligence in machines.\n        These systems are designed to think like humans and mimic their actions.\n        AI can be categorized into narrow AI and general AI.\n        '), Document(metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='\n        Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.\n        '), Document(metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='\n        Deep Learning is a subset of machine learning based on artificial neural networks.\n        It uses multiple layers to progressively extract higher-level features from raw input.\n        Deep learning has revolu

In [10]:
## text splitting
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 100,
    length_function = len,
    separators = [" "]
)

## split the documents into chunks
chunks  =text_splitter.split_documents(sample_documents)
print(chunks[0])
print(chunks[1])

page_content='Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into narrow AI and general AI.' metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}
page_content='Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised, unsupervised, and reinforcement learning.' metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}


In [12]:
print(f"Created {len(chunks)} chunks from  {len(sample_documents)} documents")
print("\nExample Chunk:")
print(f"Content: {chunks[0].page_content}")
print(f"Metadata: {chunks[0].metadata}")

Created 4 chunks from  4 documents

Example Chunk:
Content: Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into narrow AI and general AI.
Metadata: {'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}


In [13]:
## load the embedding models
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [14]:
# Initialize OpenAI embeddings with the latest model

embeddings  = OpenAIEmbeddings(
    model = "text-embedding-3-small",
    dimensions = 1536
)

## Example : create a embedding for a single text
sample_text = "what is machine learning?"
sample_embedding = embeddings.embed_query(sample_text)
sample_embedding

[-0.010711669921875,
 -0.01239776611328125,
 -0.0044708251953125,
 -0.0567626953125,
 0.0163421630859375,
 0.0033111572265625,
 -0.0023174285888671875,
 -0.00159454345703125,
 -0.034088134765625,
 0.0455322265625,
 0.01507568359375,
 -0.03662109375,
 -0.038299560546875,
 0.005764007568359375,
 0.041473388671875,
 0.037567138671875,
 0.0178375244140625,
 -0.0258941650390625,
 0.0129241943359375,
 -0.003055572509765625,
 0.0075836181640625,
 0.04864501953125,
 0.066162109375,
 -0.0308380126953125,
 0.00986480712890625,
 -0.0167236328125,
 0.01003265380859375,
 0.01690673828125,
 0.0002961158752441406,
 -0.01128387451171875,
 -0.00240325927734375,
 -0.0228424072265625,
 -0.015655517578125,
 0.0084686279296875,
 0.033233642578125,
 -0.00139617919921875,
 -0.01641845703125,
 -0.01360321044921875,
 -0.04644775390625,
 0.00862884521484375,
 -0.0770263671875,
 -0.0198974609375,
 0.0142822265625,
 0.0830078125,
 -0.044281005859375,
 -0.0220489501953125,
 -0.047760009765625,
 -0.062408447265625,

In [16]:
texts = ["AI", "Machine Learning", "Deep Learning", "Neural Network"]
batch_embeddings = embeddings.embed_documents(texts)
print(batch_embeddings[0])

[-0.0081634521484375, -0.0246124267578125, 0.0029850006103515625, 0.0251617431640625, 0.006565093994140625, -0.028228759765625, -0.005023956298828125, 0.020904541015625, -0.036895751953125, 0.01279449462890625, -0.0030364990234375, -0.020111083984375, 0.0002522468566894531, -0.03271484375, 0.0064544677734375, -0.0252685546875, -0.031097412109375, -0.054412841796875, 0.03277587890625, -0.0184173583984375, 0.01666259765625, 0.04833984375, -0.024871826171875, 0.01438140869140625, 0.0293426513671875, 0.004047393798828125, 0.00928497314453125, 0.01337432861328125, 0.0025310516357421875, -0.0225372314453125, 0.0321044921875, -0.0280303955078125, 0.005359649658203125, -0.038177490234375, -0.0167236328125, 0.01434326171875, -0.038604736328125, -0.01038360595703125, -0.0105438232421875, -0.0191650390625, 0.0321044921875, 0.014556884765625, -0.021514892578125, 0.0160675048828125, -0.01186370849609375, 0.0013990402221679688, -0.004833221435546875, -0.033660888671875, -0.02581787109375, 0.04565429

In [17]:
print(batch_embeddings[1])

[-0.022064208984375, -0.003543853759765625, -0.0191650390625, -0.034088134765625, 0.03375244140625, 0.00859832763671875, 0.0014209747314453125, 0.0260467529296875, -0.0413818359375, 0.042144775390625, -0.000701904296875, -0.037933349609375, -0.03765869140625, -0.0016508102416992188, 0.0160675048828125, 0.0164794921875, 0.0193023681640625, -0.0171051025390625, 0.0175018310546875, 0.0176849365234375, 0.0218048095703125, 0.0242767333984375, 0.0199432373046875, -0.0173187255859375, 0.03997802734375, -0.0202484130859375, 0.0291748046875, 0.040863037109375, -0.007259368896484375, -0.027191162109375, -0.01495361328125, -0.01454925537109375, -0.0380859375, -0.016510009765625, 0.0239105224609375, -0.00162506103515625, -0.0030765533447265625, 0.0267333984375, -0.04913330078125, 0.00861358642578125, -0.0289306640625, -0.0152130126953125, 0.01531982421875, 0.0736083984375, -0.017608642578125, -0.0143280029296875, -0.0299530029296875, -0.05731201171875, 0.0023956298828125, 0.06805419921875, -0.0493

In [19]:
### Compare Embeddings using cosine similarity

def compare_embeddings(text1:str, text2:str):
    """Compare semantic similarity of 2 texts using embeddings"""
    emb1 = np.array(embeddings.embed_query(text1))
    emb2 = np.array(embeddings.embed_query(text2))

    ## calculate the similarity score

    similarity  = np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2))
    return similarity

In [20]:
# Test semantic similarity
print("\nSemantic Similarity Examples:")
print(f" 'AI' vs 'Artificial Intelligence': {compare_embeddings('AI', 'Artificial Intelligence'):.3f}")


Semantic Similarity Examples:
 'AI' vs 'Artificial Intelligence': 0.563


In [21]:
print(f" 'Machine Learning' vs 'Deep Learning': {compare_embeddings('Machine Learning', 'Deep Learning'):.3f}")

 'Machine Learning' vs 'Deep Learning': 0.697


In [22]:
## Create a FAISS vector store

In [23]:
vectorstore = FAISS.from_documents(documents= chunks, embedding = embeddings)
print(f"Vector store created with {vectorstore.index.ntotal}vectors")

Vector store created with 4vectors


In [24]:
vectorstore

In [25]:
## save vector store for later use
vectorstore.save_local("faiss_index")
print("Vector store saved to 'faiss_index' directory")

Vector store saved to 'faiss_index' directory


In [27]:
## load vector store
loaded_vectorstore = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization = True
)

print(f"Loaded vector store contains {loaded_vectorstore.index.ntotal} vectors")

Loaded vector store contains 4 vectors


In [28]:
## Similarity Search
query  = "What is deep learning?"

results = vectorstore.similarity_search(query, k=3)
print(results)

[Document(id='a12319ef-6650-4b56-ae8e-33ab4f52a4dc', metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='Deep Learning is a subset of machine learning based on artificial neural networks.\n        It uses multiple layers to progressively extract higher-level features from raw input.\n        Deep learning has revolutionized computer vision, NLP, and speech recognition.'), Document(id='9b8a8b48-bbe3-4fcc-91fa-97bd93e2455a', metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.'), Document(id='8a12cefb-b756-4284-9488-0dd44ec9c5e4', metadata={'source': 'NLP Overview', 'page': 1, 'topic': 'NLP'}, page_content='Natural Language Processing (NLP) is a branch of AI that helps computers understand human lang

In [29]:
print(f"Query : {query}\n")
print("Top 3 similar chunks:")
for i, doc in enumerate(results):
    print(f"\n{i+1}. Source: {doc.metadata['source']}")
    print(f" Content: {doc.page_content[:200]}...")


Query : What is deep learning?

Top 3 similar chunks:

1. Source: Deep Learning
 Content: Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses multiple layers to progressively extract higher-level features from raw input.
        Deep learning ...

2. Source: ML Basics
 Content: Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised...

3. Source: NLP Overview
 Content: Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.
        It combines computational linguistics with machine learning and deep learning models.
      ...


In [30]:
### Similarity Search with score
results_with_scores = vectorstore.similarity_search_with_score(query, k = 3)

print("\n\nSimilarity search with scores:")
for doc, score in results_with_scores:
    print(f"\nScore : {score: .3f}")
    print(f"Source: {doc.metadata['source']}")
    print(f"Content preview: {doc.page_content[:100]}...")



Similarity search with scores:

Score :  0.513
Source: Deep Learning
Content preview: Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses m...

Score :  1.155
Source: ML Basics
Content preview: Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being...

Score :  1.249
Source: NLP Overview
Content preview: Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.
...


In [31]:
### Search with metadata filtering
filter_dict  = {"topic": "ML"}
filtered_results = vectorstore.similarity_search(query,k =3, filter= filter_dict)

print(filtered_results)

[Document(id='9b8a8b48-bbe3-4fcc-91fa-97bd93e2455a', metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.')]


In [32]:
len(filtered_results)

1

In [34]:
### Build RAG chain with LCEL

In [59]:
## LLM GROQ LLM
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

llm = init_chat_model("llama-3.1-8b-instant", model_provider="groq")
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000026119443D10>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000026119442D50>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [60]:
llm.invoke("Hi")

AIMessage(content="It's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 36, 'total_tokens': 59, 'completion_time': 0.028595297, 'completion_tokens_details': None, 'prompt_time': 0.001645843, 'prompt_tokens_details': None, 'queue_time': 0.005420469, 'total_time': 0.03024114}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_03e8423237', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019db567-0779-7d73-9550-926c284502aa-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 36, 'output_tokens': 23, 'total_tokens': 59})

In [62]:
# 1. Simple RAG chain with LCEL
simple_prompt = ChatPromptTemplate.from_template("""Answer the question based only on the following context:
Context: {context}
Question: {question}
Answer: """)

In [63]:
## Basic retriever

retriever  =vectorstore.as_retriever(
    search_type = "similarity",
    search_kwargs = {"k": 3}
)

In [64]:
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002611443D610>, search_kwargs={'k': 3})

In [65]:
# Format documents for the prompt
def format_docs(docs: List[Document]) -> str:
    """Format documents for insertion into prompt """

    formatted  = []
    for i, doc in enumerate(docs):
        source = doc.metadata.get("source", "Unknown")
        formatted.append(f"Document {i+1} (Source: {source}): \n{doc.page_content}")
    return "\n\n".join(formatted)

In [66]:
simple_rag_chain  = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | simple_prompt
    | llm
    | StrOutputParser()
)

In [67]:
simple_rag_chain

{
  context: VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002611443D610>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\nContext: {context}\nQuestion: {question}\nAnswer: '), additional_kwargs={})])
| ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000026119443D10>, async_

In [68]:
### Conversational RAG chain

conversational_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant. Use the provided context to answer questions."),
    ("placeholder", "{chat_history}"),
    ("human", "Context: {context}\n\nQuestion: {input}")
])

In [69]:
def create_conversational_rag():
    """ Create a conversational RAG chain with memory """
    return (
        RunnablePassthrough.assign(
            context  =lambda x: format_docs(retriever.invoke(x["input"]))
        )
        | conversational_prompt
        | llm
        | StrOutputParser()
    )

conversational_rag = create_conversational_rag()

In [70]:
conversational_rag

RunnableAssign(mapper={
  context: RunnableLambda(lambda x: format_docs(retriever.invoke(x['input'])))
})
| ChatPromptTemplate(input_variables=['context', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag=

In [71]:
### Streaming RAG chain
streaming_rag_chain  =(
    {"context": retriever |format_docs, "question": RunnablePassthrough()}
    | simple_prompt
    | llm
)

print("Modern RAG chains created successfully!")
print("Available chains:")
print("-simple_rag_chain : Basic Q&A")
print("-conversationa_rag : Maintains conversation history")
print("-streaming_rag_chain: Supports token streaming")

Modern RAG chains created successfully!
Available chains:
-simple_rag_chain : Basic Q&A
-conversationa_rag : Maintains conversation history
-streaming_rag_chain: Supports token streaming


In [75]:
# Test function for different chain types
def test_rag_chains(question: str):
    """ Test all RAG chinas variants"""
    print(f"Question: {question}")
    print("=" *80)

    #1. Simple RAG
    print("\n1. Simple RAG chain:")
    answer  =simple_rag_chain.invoke(question)
    print(f"Answer: {answer}")


    print("\n2. Streaming RAG:")
    print("Answer: ", end="", flush = True)
    for chunk in streaming_rag_chain.stream(question):
        print(chunk.content, end="", flush=True)
    print()

In [76]:
test_rag_chains("What is difference between AI and machine learning?")

Question: What is difference between AI and machine learning?

1. Simple RAG chain:
Answer: Based on the given context, the main difference between AI and machine learning is that AI is the broader field of creating machines that can think and act like humans, while machine learning is a subset of AI that specifically focuses on enabling systems to learn from data.

In other words, AI is the overall goal of creating intelligent machines, and machine learning is a key technology used to achieve that goal by allowing systems to learn from data and improve their performance over time.

2. Streaming RAG:
Answer: Based on the provided context, the main difference between AI and Machine Learning (ML) is that AI is a broader field that aims to create systems that think like humans and mimic their actions, while ML is a subset of AI that focuses on enabling systems to learn from data.

In other words, AI is an umbrella term that encompasses various technologies, including ML, which is a specif

In [77]:
# Test with multipe question
test_questions = [
    "What is difference between AI and machine learning?",
    "Explain deep learning in simple terms",
    "How does NLP work?",
]

for question in test_questions:
    print("\n" + "=" *80 + "\n")
    test_rag_chains(question)




Question: What is difference between AI and machine learning?

1. Simple RAG chain:
Answer: Based on the provided context, the main difference between AI and Machine Learning is that Artificial Intelligence is the broader field that aims to simulate human intelligence in machines, while Machine Learning is a subset of AI that specifically focuses on enabling systems to learn from data.

In other words, AI is the umbrella term that encompasses various techniques and approaches to create intelligent machines, while Machine Learning is one of the key techniques used within AI to enable systems to learn and improve from experience.

2. Streaming RAG:
Answer: Based on the given context, the main difference between AI and Machine Learning (ML) is that AI is a broader field that focuses on simulating human intelligence in machines, while ML is a subset of AI that specifically enables systems to learn from data.

In other words, AI is a more general term that encompasses various approaches t

In [78]:
## Conversational example
print("\n3. Conversational RAG Example:")
chat_history = []

#First question
q1 = "What is machine learning?"
a1 = conversational_rag.invoke({
    "input": q1,
    "chat_history": chat_history
})

print(f"Q1 : {q1}")
print(f"A1: {a1}")


3. Conversational RAG Example:
Q1 : What is machine learning?
A1: According to Document 1 (Source: ML Basics), machine learning is a subset of AI that enables systems to learn from data, instead of being explicitly programmed.


In [79]:
# update history
chat_history.extend([
    HumanMessage(content = q1),
    AIMessage(content = a1)
])

In [80]:
# Follow up question
q2 = "How is it different from traditional programming?"
a2 = conversational_rag.invoke({
    "input" : q2,
    "chat_history": chat_history
})

print(f"\nQ2 : {q2}")
print(f"A2: {a2}")


Q2 : How is it different from traditional programming?
A2: Based on the provided context, machine learning is different from traditional programming in the following ways:

1. **Learning from data**: Machine learning algorithms find patterns in data instead of being explicitly programmed to perform a specific task. Traditional programming involves writing explicit code for every step of the process.

2. **Pattern recognition**: Machine learning enables systems to recognize patterns in data, whereas traditional programming relies on explicit instructions.

The key difference lies in the approach: traditional programming focuses on explicit instructions, while machine learning focuses on learning from data and patterns.

(References: Document 1, Document 2)
